# 04 · Feature Engineering — physical_itens_venda_caixa
**Squad 3 – Batch | Alexandro Ferreira da Silva**

Criação de features derivadas a partir da tabela de itens de venda no caixa físico.

### Tabelas de entrada (Unity Catalog)
| Tabela | Linhas | Papel |
|--------|--------|-------|
| `workspace.squad3.physical_itens_venda_caixa` | 3.562.643 | tabela principal |
| `workspace.squad3.physical_vendas_caixa` | 375.000 | contexto da transação |
| `workspace.squad3.physical_lojas` | 29 | dados da loja |

### Regras e KPIs gerados por este notebook
Nenhuma regra da Silver nem KPI da Gold: este notebook é exploratório e gera apenas as features abaixo,
a partir das cópias derivadas da RAW no Unity Catalog.

### Features geradas
| Feature | Lógica |
|---------|--------|
| `categoria_produto` | 1ª parte do código de barras |
| `marca_produto` | 2ª parte do código de barras |
| `tamanho_embalagem` | 3ª parte do código de barras |
| `flag_produto_pesavel` | quantidade não-inteira → vendido por peso |
| `qtd_itens_por_transacao` | agregação por id_transacao |
| `qtd_produtos_distintos` | variedade de produtos por transação |
| `valor_total_transacao` | soma por id_transacao |
| `ticket_medio_item` | valor_total / qtd_itens |

### Tabela de saída
`workspace.squad3.physical_itens_venda_caixa_features`

## 1. Leitura das tabelas
Lê do Unity Catalog a tabela de itens e as duas tabelas de contexto (vendas e lojas).

In [0]:
# Célula 1 — Leitura das tabelas

# Leitura das tabelas do Unity Catalog
df_itens   = spark.table("workspace.squad3.physical_itens_venda_caixa")
df_vendas  = spark.table("workspace.squad3.physical_vendas_caixa")
df_lojas   = spark.table("workspace.squad3.physical_lojas")

print(f"✅ physical_itens_venda_caixa  : {df_itens.count():>10,} linhas | {len(df_itens.columns)} colunas")
print(f"✅ physical_vendas_caixa       : {df_vendas.count():>10,} linhas | {len(df_vendas.columns)} colunas")
print(f"✅ physical_lojas              : {df_lojas.count():>10,} linhas | {len(df_lojas.columns)} colunas")

## 2. Parse do código de barras
O `codigo_barras_produto` segue o padrão `categoria-marca-tamanho-variante`. As três primeiras
partes viram `categoria_produto`, `marca_produto` e `tamanho_embalagem`.

In [0]:
# Célula 2 — Parse do código de barras

from pyspark.sql import functions as F

# O código de barras segue o padrão: categoria-marca-tamanho-variante
# Exemplo: "sorvetes-oggi-2l-napolitanopote"
df_parsed = df_itens.withColumn(
    "partes", F.split(F.col("codigo_barras_produto"), "-")
).withColumn(
    "categoria_produto", F.col("partes")[0]
).withColumn(
    "marca_produto", F.col("partes")[1]
).withColumn(
    "tamanho_embalagem", F.col("partes")[2]
).drop("partes")

print("✅ Parse do código de barras concluído.")
df_parsed.select("codigo_barras_produto", "categoria_produto", "marca_produto", "tamanho_embalagem").show(5, truncate=False)

## 3. Feature de produto pesável
Marca os itens com quantidade fracionada, que são os produtos vendidos por peso ou volume (kg ou L).
Não há features de desconto: a diferença entre `valor_total_item` e quantidade × preço é apenas
arredondamento da quantidade, e não desconto.

In [0]:
# Célula 3 — Feature de produto pesável
# Não há features de desconto: a diferença entre valor_total_item e quantidade × preço
# é só arredondamento da quantidade (simétrica, de R$ 0,03 a R$ 0,15), e não desconto.

df_features = df_parsed.withColumn(
    # Produto vendido por peso/volume: quantidade com parte decimal significativa
    "flag_produto_pesavel", F.when(
        (F.col("quantidade") % 1) > 0.01, 1
    ).otherwise(0)
)

print("✅ Feature de produto pesável criada.")
df_features.select(
    "id_item_venda", "quantidade", "preco_unitario_registro",
    "valor_total_item", "flag_produto_pesavel"
).show(5)

## 4. Agregações por transação
Calcula por `id_transacao` a quantidade de itens, a variedade de produtos, o valor total e o valor
médio por item, e junta o resultado de volta em cada linha de item.

In [0]:
# Célula 4 — Agregações por transação

# Métricas agregadas no nível da transação
df_agg_transacao = df_features.groupBy("id_transacao").agg(
    F.count("id_item_venda").alias("qtd_itens_por_transacao"),
    F.countDistinct("codigo_barras_produto").alias("qtd_produtos_distintos"),
    F.round(F.sum("valor_total_item"), 2).alias("valor_total_transacao"),
    F.round(F.avg("valor_total_item"), 2).alias("ticket_medio_item"),
    F.max("flag_produto_pesavel").alias("flag_transacao_com_pesavel"),
)

# Join das agregações de volta na tabela de itens
df_features = df_features.join(df_agg_transacao, on="id_transacao", how="left")

print("✅ Agregações por transação criadas.")
df_features.select(
    "id_transacao", "qtd_itens_por_transacao",
    "qtd_produtos_distintos", "valor_total_transacao", "ticket_medio_item"
).show(5)

## 5. Contexto da venda
Traz de `physical_vendas_caixa` os dados da transação (loja, caixa, operador, data, pagamento).

In [0]:
# Célula 5 — Join com physical_vendas_caixa e physical_lojas

# Seleciona colunas relevantes de vendas (evita duplicar colunas que já existem)
colunas_vendas = [c for c in df_vendas.columns if c != "id_transacao"]
df_vendas_sel = df_vendas.select("id_transacao", *colunas_vendas)

# Join com vendas para trazer contexto da transação (data, hora, loja)
df_features = df_features.join(df_vendas_sel, on="id_transacao", how="left")

# Verifica se physical_lojas tem coluna de join compatível com vendas
print("Colunas de physical_lojas:", df_lojas.columns)
print("Colunas de physical_vendas_caixa:", df_vendas.columns)

## 6. Contexto da loja
Enriquece cada item com nome, cidade e estado da loja, a partir de `physical_lojas`.

In [0]:
# Célula 5b — Join com physical_lojas (enriquecimento por loja)

df_lojas_sel = df_lojas.select("id_loja", "nome_loja", "cidade_loja", "estado_loja", "peso_vendas")

df_features = df_features.join(df_lojas_sel, on="id_loja", how="left")

print("✅ Join com physical_lojas concluído.")
df_features.select("id_loja", "nome_loja", "cidade_loja", "estado_loja").show(3)

## 7. Gravação da tabela de features
Grava `workspace.squad3.physical_itens_venda_caixa_features` e lista as colunas geradas.

In [0]:
# Célula 6 — Salvar tabela de features no Unity Catalog

(df_features
    .write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.squad3.physical_itens_venda_caixa_features"))

total = df_features.count()
print(f"✅ workspace.squad3.physical_itens_venda_caixa_features")
print(f"   {total:,} linhas | {len(df_features.columns)} colunas")
print(f"\n📋 Colunas geradas:")
for col in df_features.columns:
    print(f"   • {col}")

## 8. Validação das features
Confere as categorias mais frequentes, o percentual de itens pesáveis e as médias por transação.
As médias são calculadas sobre linhas de item, então transações maiores pesam mais.

In [0]:
# Célula 7 — Validação das features

print("=" * 60)
print("VALIDAÇÃO DAS FEATURES")
print("=" * 60)

total = df_features.count()

# Taxa de itens pesáveis
pct_pesavel  = df_features.filter(F.col("flag_produto_pesavel") == 1).count() / total * 100

# Categorias mais frequentes
print("\n📦 Top 5 categorias de produto:")
df_features.groupBy("categoria_produto").count() \
    .orderBy(F.desc("count")).show(5, truncate=False)

print(f"\n⚖️  Itens vendidos por peso: {pct_pesavel:.1f}%")

print(f"\n💰 Ticket médio por item  : R$ {df_features.agg(F.avg('ticket_medio_item')).collect()[0][0]:.2f}")
print(f"🛒 Média de itens/transação: {df_features.agg(F.avg('qtd_itens_por_transacao')).collect()[0][0]:.1f}")

print("\n✅ Feature Engineering concluído!")